# 07 - Model Explainability

**Goal:** Understand which factors drive defect predictions, using logistic regression coefficients, tree feature importance and permutation importance.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv")["defect"]

lr = joblib.load("../models/logistic_regression.joblib")
rf = joblib.load("../models/random_forest.joblib")

In [ ]:
coef = lr.named_steps["model"].coef_[0]     
coef_df = pd.DataFrame({
    "feature": X_test.columns,
    "coefficient": coef,
    "odds_ratio": np.exp(coef)
}).sort_values("coefficient", ascending=False).round(3)
coef_df

In [ ]:
plt.figure(figsize=(8, 5))
coef_df.sort_values("coefficient").plot(
    x="feature", y="coefficient", kind="barh", legend=False, color="steelblue")
plt.title("Logistic Regression Coefficients (scaled features)")
plt.axvline(0, color="black", linewidth=0.8)
plt.savefig("../reports/11_lr_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X_test.columns).sort_values()

plt.figure(figsize=(8, 5))
imp.plot(kind="barh", color="darkorange")
plt.title("Random Forest Feature Importance")
plt.savefig("../reports/12_rf_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(lr, X_test, y_test, scoring="roc_auc",
                                n_repeats=20, random_state=42)

perm = pd.Series(result.importances_mean, index=X_test.columns).sort_values()

plt.figure(figsize=(8, 5))
perm.plot(kind="barh", xerr=pd.Series(result.importances_std, index=X_test.columns)[perm.index],
          color="seagreen")
plt.xlabel("Drop in ROC-AUC when shuffled")
plt.title("Permutation Importance (Logistic Regression)")
plt.savefig("../reports/13_permutation_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## Findings
- **Top drivers (permutation importance):** cutting temperature, vibration and tool wear. This matches the logic used to generate the data and also machining physics.
- **Coolant flow** has a negative coefficient: more coolant → lower defect risk.
- `cutting_intensity` and `heat_per_coolant` also showed negative coefficients. This is not physically intuitive and is caused by **multicollinearity** with temperature and vibration.

## Limitations
- Coefficients of correlated features are unstable, so overall patterns are more reliable than single values.
- Permutation importance can understate correlated features because the others compensate.